# Comparación de Formatos de Export SISAV2

## Resumen ejecutivo

La plataforma SISAV2 actualmente entrega datos en dos formatos distintos: un **formato legacy** (36 columnas, usado entre 2016-2024) y un **formato expandido** (48-49 columnas, disponible desde 2025). El formato expandido incluye información clave para acreditación que el legacy no tiene: **participantes desagregados por género**, **asignaturas y cátedras asociadas**, **logros de aprendizaje** y el **ciclo del modelo educativo**. Sin estos datos, es imposible construir indicadores de impacto curricular o reportar participación con enfoque de género. Solicitar los datos históricos en formato expandido -o al menos los campos adicionales- permitiría construir un dashboard de VcM con cobertura completa desde 2016.

---

In [1]:
import sys
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / ".."]
PROJECT_ROOT = next((p.resolve() for p in _candidates if (p / "src" / "__init__.py").exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import leer_excels
from src.schema import COLUMNAS_CANONICAS

sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

## 1. Carga de datos

### 1.1 Formato legacy (36 columnas)
Archivos procesados por el pipeline actual (`src.ingest.leer_excels`).

In [2]:
DATA_DIR = Path("/Users/ignacioramirez/Desktop/Proyecto VCM/drive-download-20260623T174348Z-3-001/iniciativas_a_excel/")

df_legacy, _ = leer_excels(DATA_DIR, strict=True)
print(f"Legacy: {df_legacy.shape[0]} filas × {len(COLUMNAS_CANONICAS)} columnas ({df_legacy['_archivo_origen'].nunique()} archivos)")

FileNotFoundError: No se encontraron archivos .xlsx en \Users\ignacioramirez\Desktop\Proyecto VCM\drive-download-20260623T174348Z-3-001\iniciativas_a_excel

### 1.2 Formato expandido (48-49 columnas)
Los tres archivos de convocatorias 2025 con esquema nuevo. Se lee la hoja `Postulaciones`.

In [ ]:
ARCHIVOS_EXPANDIDOS = {
    "Extensión Académica 2025": DATA_DIR / "PRE_GRADO__conv32__Convocatoria_proyecto_VcM_Extensión_Académica_2025.xlsx",
    "VEDP 2025": DATA_DIR / "PRE_GRADO__conv39__Convocatoria_proyecto_Vinculación_con_el_Entorno_Disciplinar_Profesional_2025.xlsx",
    "Vinculación con Titulados 2025": DATA_DIR / "PRE_GRADO__conv38__Convocatoria_proyecto_VcM_Vinculación_con_Titulados_2025.xlsx",
}

frames_exp = {}
for nombre, ruta in ARCHIVOS_EXPANDIDOS.items():
    df_exp = pd.read_excel(ruta, sheet_name="Postulaciones", engine="openpyxl")
    frames_exp[nombre] = df_exp
    print(f"{nombre}: {df_exp.shape[0]} filas × {df_exp.shape[1]} columnas")

# Consolidar para análisis de cobertura
df_expandido = pd.concat(frames_exp.values(), ignore_index=True)
print(f"\nTotal expandido: {df_expandido.shape[0]} filas × {df_expandido.shape[1]} columnas")

---
## 2. Inventario de columnas

### 2.1 Formato legacy

In [ ]:
legacy_cov = df_legacy[COLUMNAS_CANONICAS].notna().mean() * 100
inv_legacy = pd.DataFrame({
    "Columna": COLUMNAS_CANONICAS,
    "% Poblado": legacy_cov.values.round(1),
})
print(f"Formato legacy: {len(COLUMNAS_CANONICAS)} columnas")
print(f"Columnas con dato (>0%): {(legacy_cov > 0).sum()}")
print(f"Columnas 100% vacías: {(legacy_cov == 0).sum()}")
print()
print(inv_legacy.to_string(index=False))

### 2.2 Formato expandido

In [ ]:
exp_cov = df_expandido.notna().mean() * 100
inv_exp = pd.DataFrame({
    "Columna": df_expandido.columns,
    "% Poblado": exp_cov.values.round(1),
})
print(f"Formato expandido: {len(df_expandido.columns)} columnas")
print(f"Columnas con dato (>0%): {(exp_cov > 0).sum()}")
print(f"Columnas 100% vacías: {(exp_cov == 0).sum()}")
print()
print(inv_exp.to_string(index=False))

---
## 3. Comparación lado a lado: mapeo de columnas

Mapeo construido por inspección manual de ambos esquemas. Se clasifica cada columna en:
- **Equivalente**: existe en ambos formatos (quizás con nombre distinto)
- **Solo expandido**: columna nueva sin contraparte en legacy
- **Solo legacy**: columna del formato antiguo sin equivalente en el nuevo

In [ ]:
# Mapeo de equivalencias: legacy → expandido
MAPEO_EQUIVALENCIAS = {
    "Codigo": "Código / CÓDIGO SISAV / Código SISAV",
    "Estado SISAV": "Estado SISAV",
    "Facultad": "Facultad",
    "Carrera": "Carrera",
    "Nombre de la iniciativa": "Nombre de iniciativa",
    "Dominios disciplinares": "Dominios disciplinares",
    "Área": "Área",
    "Actividad": "Actividad",
    "Competencia Sello": "Competencia sello",
    "Contribución Interna": "Contribución interna",
    "Contribución Externa": "Contribución externa",
    "Competencia genérica": "Competencias génericas",
    "PDI": "PDI",
    "ODS": "ODS",
    "Modalidad": "Modalidad",
    "Semestre Ejecución": "Semestre de ejecución de la iniciativa",
    "Cantidad Act Planificadas": "Cantidad de actividades planificadas",
    "Fecha Inicio": "Fecha de inicio",
    "Fecha Termino": "Fecha de finalización",
    "Grupo de interés": "Grupo de interés",
    "Área de influencia": "Área de influencia",
    "Tipo de requerimientos": "Tipo de requerimientos",
    "Monto": "Monto",
    "Evidencia": "Evidencia / Existe de evidencia / Existe informe de evidencia",
}

# Columnas SOLO en el formato expandido (no tienen contraparte en legacy)
SOLO_EXPANDIDO = [
    "Académico postulante",
    "Objetivo de la iniciativa",
    "Ciclo del modelo educativo / Ciclo / Línea de vinculación",
    "Logros de aprendizaje",
    "Tipo de actividad",
    "Nombre de asignatura asociada",
    "Código cátedra asociada",
    "Semestre de la cátedra",
    "N de estudiantes en cátedra asociada",
    "N de estudiantes (M)",
    "N de estudiantes (F)",
    "N de estudiantes (Otro)",
    "N de titulados (M)",
    "N de titulados (F)",
    "N de titulados (Otro)",
    "N de académicos (M)",
    "N de académicos (F)",
    "N de académicos (Otro)",
    "N de funcionarios (M)",
    "N de funcionarios (F)",
    "N de funcionarios (Otro)",
    "N de personas relacionadas a instituciones externas (M)",
    "N de personas relacionadas a instituciones externas (F)",
    "N de personas relacionadas a instituciones externas (Otro)",
    "N total de participantes de la iniciativa",
    "Instituciones externas",
]

# Columnas SOLO en legacy (no tienen contraparte en el expandido)
SOLO_LEGACY = [
    "N",
    "PLATAFORMA",
    "UA",
    "Dimensión",
    "Perfil del entorno disciplinar y/o profesional",
    "Objetivo específico electivo",
    "N estudiantes",
    "N Docentes",
    "N Titulados",
    "N Instituciones del medio externo participantes",
    "Total",
    "Comentarios",
]

print(f"Columnas equivalentes (en ambos formatos): {len(MAPEO_EQUIVALENCIAS)}")
print(f"Columnas solo en expandido (nuevas):       {len(SOLO_EXPANDIDO)}")
print(f"Columnas solo en legacy (sin equivalente): {len(SOLO_LEGACY)}")
print()
print("=" * 90)
print(f"{'LEGACY':<45s} {'EXPANDIDO':<45s}")
print("=" * 90)
print("--- EQUIVALENTES ---")
for leg, exp in MAPEO_EQUIVALENCIAS.items():
    print(f"  {leg:<43s} ↔ {exp}")
print()
print("--- SOLO EN EXPANDIDO (NUEVAS) ---")
for col in SOLO_EXPANDIDO:
    print(f"  {'-':<43s}   {col}")
print()
print("--- SOLO EN LEGACY ---")
for col in SOLO_LEGACY:
    print(f"  {col:<43s}   -")

---
## 4. Argumento central: cobertura comparada en columnas clave

Se compara el porcentaje de poblado para indicadores relevantes de gestión y acreditación.
Los campos con desagregación por género y los curriculares son los que más diferencia muestran.

In [ ]:
# Indicadores clave y su nombre en cada formato
INDICADORES_CLAVE = {
    "Participantes desagregados\n(género M/F/Otro)": {
        "legacy": ["N estudiantes", "N Docentes", "N Titulados"],
        "expandido": ["N de estudiantes (M)", "N de estudiantes (F)", "N de estudiantes (Otro)",
                      "N de titulados (M)", "N de titulados (F)", "N de titulados (Otro)",
                      "N de académicos (M)", "N de académicos (F)", "N de académicos (Otro)"],
    },
    "Asignatura/cátedra\nasociada": {
        "legacy": [],
        "expandido": ["Nombre de asignatura asociada", "Código cátedra asociada"],
    },
    "Logros de\naprendizaje": {
        "legacy": [],
        "expandido": ["Logros de aprendizaje"],
    },
    "Ciclo modelo\neducativo": {
        "legacy": [],
        "expandido": ["Ciclo del modelo educativo", "Ciclo"],
    },
    "Contribución\ninterna/externa": {
        "legacy": ["Contribución Interna", "Contribución Externa"],
        "expandido": ["Contribución interna", "Contribución externa"],
    },
    "ODS": {
        "legacy": ["ODS"],
        "expandido": ["ODS"],
    },
    "Competencias": {
        "legacy": ["Competencia Sello", "Competencia genérica"],
        "expandido": ["Competencia sello", "Competencias génericas"],
    },
    "Monto": {
        "legacy": ["Monto"],
        "expandido": ["Monto"],
    },
    "Total\nparticipantes": {
        "legacy": ["Total"],
        "expandido": ["N total de participantes de la iniciativa"],
    },
}

def cobertura_promedio(df, cols):
    """Promedio de % poblado de las columnas que existen en el df."""
    presentes = [c for c in cols if c in df.columns]
    if not presentes:
        return 0.0
    return df[presentes].notna().mean().mean() * 100

comparacion = []
for indicador, fuentes in INDICADORES_CLAVE.items():
    pct_legacy = cobertura_promedio(df_legacy, fuentes["legacy"])
    pct_exp = cobertura_promedio(df_expandido, fuentes["expandido"])
    comparacion.append({"Indicador": indicador, "Legacy (%)": round(pct_legacy, 1), "Expandido (%)": round(pct_exp, 1)})

df_comp = pd.DataFrame(comparacion)
print(df_comp.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

y = range(len(df_comp))
bar_height = 0.35

bars1 = ax.barh([i + bar_height/2 for i in y], df_comp["Legacy (%)"], bar_height,
               label="Legacy (36 cols)", color="#ef5350", alpha=0.85)
bars2 = ax.barh([i - bar_height/2 for i in y], df_comp["Expandido (%)"], bar_height,
               label="Expandido (48-49 cols)", color="#42a5f5", alpha=0.85)

ax.set_yticks(list(y))
ax.set_yticklabels(df_comp["Indicador"].str.replace("\n", " "))
ax.set_xlabel("% de celdas pobladas")
ax.set_title("Cobertura de indicadores clave: Legacy vs Expandido", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.set_xlim(0, 110)

for bar in bars1:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.0f}%", va="center", fontsize=9)
for bar in bars2:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.0f}%", va="center", fontsize=9)

ax.axvline(x=80, color="gray", linestyle="--", alpha=0.5, label="Umbral útil (80%)")
plt.tight_layout()
plt.show()

---
## 5. Indicadores que SOLO el formato expandido habilita

Estas son capacidades analíticas **imposibles** con el formato legacy, que requieren columnas exclusivas del formato expandido.

### 5.1 Participación desagregada por género

El formato expandido reporta participantes en 5 categorías (estudiantes, titulados, académicos, funcionarios, personas de instituciones externas), cada una desagregada en M/F/Otro. Esto habilita:

- Indicadores de equidad de género en VcM
- Reportes para acreditación con enfoque de género (estándar CNA)
- Análisis de brechas por facultad, instrumento y año

El formato legacy solo tiene `N estudiantes`, `N Docentes`, `N Titulados` - **todos 100% vacíos** en las 33 convocatorias procesadas.

In [ ]:
# Ejemplo real (anonimizado) de datos desagregados por género
cols_genero = [c for c in df_expandido.columns if "(M)" in c or "(F)" in c or "(Otro)" in c]
muestra_genero = df_expandido[cols_genero].dropna(how="all").head(5)

print("Muestra de participación desagregada (formato expandido):")
print(muestra_genero.to_string(index=False))
print(f"\nFilas con datos de género: {df_expandido[cols_genero].dropna(how='all').shape[0]} de {len(df_expandido)} ({df_expandido[cols_genero].dropna(how='all').shape[0]/len(df_expandido)*100:.0f}%)")

# Comparar con legacy
cols_part_legacy = ["N estudiantes", "N Docentes", "N Titulados", "Total"]
poblado_legacy = df_legacy[cols_part_legacy].notna().any(axis=1).sum()
print(f"\nEn legacy, filas con ALGÚN dato de participantes: {poblado_legacy} de {len(df_legacy)} ({poblado_legacy/len(df_legacy)*100:.0f}%)")

### 5.2 Contribución al plan de estudio (asignatura + logros de aprendizaje)

El formato expandido vincula cada iniciativa a una **asignatura/cátedra específica** y sus **logros de aprendizaje**. Esto permite:

- Medir la contribución de VcM al currículo (¿cuántas asignaturas integran vinculación?)
- Mapear cobertura de logros de aprendizaje por carrera
- Evaluar articulación del modelo educativo con la vinculación

En el formato legacy estas columnas **no existen**.

In [ ]:
cols_curricular = ["Nombre de asignatura asociada", "Código cátedra asociada", "Semestre de la cátedra"]
cols_curr_presentes = [c for c in cols_curricular if c in df_expandido.columns]

muestra_curr = df_expandido[cols_curr_presentes].dropna(how="all").head(8)
print("Muestra de vinculación curricular (formato expandido):")
print(muestra_curr.to_string(index=False))

n_con_asig = df_expandido["Nombre de asignatura asociada"].notna().sum()
print(f"\nIniciativas vinculadas a una asignatura: {n_con_asig} de {len(df_expandido)} ({n_con_asig/len(df_expandido)*100:.0f}%)")

# Logros de aprendizaje (solo en VEDP)
if "Logros de aprendizaje" in df_expandido.columns:
    n_logros = df_expandido["Logros de aprendizaje"].notna().sum()
    print(f"Iniciativas con logros de aprendizaje declarados: {n_logros} de {len(df_expandido)} ({n_logros/len(df_expandido)*100:.0f}%)")
    print(f"\nEjemplo de logros de aprendizaje (primeros 3):")
    ejemplos = df_expandido["Logros de aprendizaje"].dropna().head(3)
    for i, val in enumerate(ejemplos, 1):
        print(f"  {i}. {str(val)[:120]}{'...' if len(str(val))>120 else ''}")

### 5.3 Ciclo del modelo educativo

El formato expandido clasifica cada iniciativa según el **ciclo del modelo educativo** institucional (ej. ciclo inicial, de desarrollo, de síntesis). Esto permite:

- Evaluar distribución de VcM a lo largo de la trayectoria formativa
- Detectar concentraciones o vacíos en ciclos específicos
- Reportar coherencia curricular para acreditación

In [ ]:
# El campo se llama distinto según el instrumento
col_ciclo = None
for c in ["Ciclo del modelo educativo", "Ciclo", "Línea de vinculación"]:
    if c in df_expandido.columns:
        col_ciclo = c
        break

if col_ciclo:
    dist_ciclo = df_expandido[col_ciclo].value_counts(dropna=False)
    print(f"Distribución de '{col_ciclo}' en formato expandido:")
    print(dist_ciclo.to_string())
else:
    # Buscar en cada frame individual
    for nombre, df_f in frames_exp.items():
        for c in ["Ciclo del modelo educativo", "Ciclo", "Línea de vinculación"]:
            if c in df_f.columns:
                print(f"{nombre} → '{c}':")
                print(df_f[c].value_counts(dropna=False).to_string())
                print()

### 5.4 Resumen: KPIs habilitados solo por el formato expandido

| KPI | Requiere | Disponible en Legacy | Disponible en Expandido |
|-----|----------|---------------------|------------------------|
| Participantes por género (brecha M/F) | Conteos desagregados M/F/Otro | No (columnas vacías) | Sí (82-91% poblado) |
| Cobertura curricular de VcM | Asignatura + cátedra asociada | No (no existe) | Sí (50-100% poblado) |
| Logros de aprendizaje por iniciativa | Campo Logros de aprendizaje | No (no existe) | Sí (solo VEDP, 100%) |
| Distribución por ciclo formativo | Ciclo del modelo educativo | No (no existe) | Sí (100% poblado) |
| Participación de instituciones externas | Nombre de instituciones | No (solo conteo vacío) | Sí (37-67% poblado) |
| Académico responsable por iniciativa | Académico postulante | No (no existe) | Sí (100%) |

---
## 6. Conclusión y solicitud

### Qué se pierde con el formato legacy

- **No es posible reportar participación con enfoque de género** (requisito CNA). Las columnas de conteo del legacy (N estudiantes, N Docentes, N Titulados, Total) están 100% vacías en las 33 convocatorias 2016-2024.
- **No se puede medir la contribución de VcM al currículo**: no existe vinculación a asignatura, cátedra ni logros de aprendizaje.
- **No se puede evaluar la distribución de VcM por ciclo formativo**: el campo no existe.
- **No se puede identificar al académico responsable** de cada iniciativa.

### Qué se gana con el formato expandido

- **Indicadores de género completos** con desagregación en 5 categorías × 3 géneros.
- **Articulación curricular medible**: cada iniciativa ligada a asignatura, cátedra y logros de aprendizaje.
- **Trazabilidad del modelo educativo** vía ciclo formativo.
- **Identificación de responsables** para seguimiento y evaluación.
- **Compatibilidad futura**: el formato expandido es el estándar actual de SISAV2; migrar ahora evita deuda técnica.

### Solicitud a VcM

Se solicita que los datos históricos (2016-2024) se re-exporten desde SISAV2 en el formato expandido, o al menos que se complementen con los campos adicionales de participación desagregada y vinculación curricular. Esto permitiría construir un dashboard con series temporales completas y comparables, sin puntos ciegos en los indicadores de acreditación.

Si la re-exportación no es viable, se solicita como mínimo que **todas las entregas futuras** se realicen exclusivamente en formato expandido, para que el pipeline pueda incorporarlos sin pérdida de información.